# Phase 1 — Data Ingestion

This notebook handles the initial data pipeline:
1. Downloads raw data from **OPSD** (Energy), **SMARD** (Market), and **Meteostat** (Weather)
2. Synchronizes timezones and resamples to hourly frequency
3. Merges all sources into a single `master.parquet` file

**Note**: If `force=False`, the script will skip downloading sources that already exist in `data/raw`.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
from pathlib import Path
from src.ingest.merge import build_master

# Configure logging to track download progress
logging.basicConfig(
    level=logging.INFO, 
    format='%(levelname)s %(name)s: %(message)s'
)

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Directories initialized.")

## 1. Execute Pipeline
This may take several minutes depending on your internet connection and the `start` date.

In [ ]:
master = build_master(
    raw_dir=RAW,
    processed_dir=PROCESSED,
    start='2015-01-01',
    end=None,    # Defaults to latest available
    force=False  # Set to True to re-download everything
)

if master is not None:
    print(f'\n✓ Phase 1 complete.')
    print(f'Dataset: {master.index.min()} to {master.index.max()}')
    print(f'Shape:   {master.shape[0]:,} rows × {master.shape[1]} columns')
else:
    print("Build failed. Check logs for details.")

## 2. Quick Data Audit
Check for missing values (NaNs) across the merged columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 4))
sns.heatmap(master.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Map (Yellow = Missing)')
plt.show()

print("\nMissing values per column:")
print(master.isnull().sum()[master.isnull().sum() > 0])